# 07 — Scenario: High-Value Customer Opportunity

**The situation:** the Milan Flagship Private Trunk Show campaign produced a
cohort of customers who show unusually high repeat-purchase behavior and strong
affinity for the new Portofino capsule collection.

This notebook proves: cohort discovery, next-best-product / next-best-client
scoring, and expected incremental revenue from targeted advisor outreach.

In [1]:
import sys
sys.path.insert(0, "../src")
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from retail_synth.viz import CATEGORICAL, SEQUENTIAL_BLUE, DIVERGING, STATUS, style_fig

pd.options.display.float_format = "{:,.1f}".format
con = duckdb.connect("../data/warehouse/retail.duckdb", read_only=True)
AS_OF = con.execute("SELECT week_start_date FROM silver.dim_week WHERE is_as_of_week").fetchone()[0]
print(f"Connected. AS_OF date = {AS_OF}")

Connected. AS_OF date = 2025-12-01


## Investigate — repeat-purchase lift vs. a matched control

At real sales volume, a binary "made >=2 purchases" flag saturates near 100% for
almost every active customer — it stops being able to tell anyone apart. The
signal shows up clearly in *how many times* someone repeat-purchases, so that's
the metric we compare: average order count, cohort vs. a matched control of
similarly high-tier non-cohort customers.

In [2]:
coh = con.execute("""
    SELECT is_milan_cohort, loyalty_tier, COUNT(*) n, AVG(n_orders) avg_orders
    FROM gold.customer_cohort_scorecard
    WHERE home_region = 'ITA' AND loyalty_tier IN ('Gold','Platinum','Private Client')
    GROUP BY 1, 2
""").df()
coh["weighted"] = coh["avg_orders"] * coh["n"]
summary = coh.groupby("is_milan_cohort", as_index=False).agg(weighted=("weighted", "sum"), n=("n", "sum"))
summary["avg_orders"] = summary["weighted"] / summary["n"]
lift = summary.loc[summary.is_milan_cohort, "avg_orders"].iloc[0] / summary.loc[~summary.is_milan_cohort, "avg_orders"].iloc[0]
print(f"Milan cohort repeat-purchase lift vs. matched control: {lift:.1f}x")

fig = px.bar(summary, x="is_milan_cohort", y="avg_orders", color="is_milan_cohort",
             color_discrete_sequence=[CATEGORICAL[2], CATEGORICAL[0]],
             labels={"is_milan_cohort": "Milan trunk-show cohort", "avg_orders": "Average orders per customer"})
fig.update_layout(showlegend=False)
style_fig(fig, "Average orders per customer: Milan cohort vs. matched Italy control")

Milan cohort repeat-purchase lift vs. matched control: 2.6x


## Investigate — capsule affinity

In [3]:
capsule = con.execute("""
    SELECT is_milan_cohort, AVG(capsule_units) avg_capsule_units
    FROM gold.customer_cohort_scorecard WHERE home_region = 'ITA' GROUP BY 1
""").df()
capsule

,is_milan_cohort,avg_capsule_units
0,True,10.0
1,False,0.7


## Simulate — next-best-client ranking

The propensity target is "above-median order frequency for this population" —
`is_repeat_purchaser` is too saturated at this volume to be a useful classification
target (see above), so we define a sharper one on the fly.

In [4]:
from sklearn.linear_model import LogisticRegression

model_df = con.execute("""
    SELECT customer_id, is_milan_cohort, loyalty_tier, total_units, total_revenue, capsule_units, n_orders
    FROM gold.customer_cohort_scorecard WHERE home_region = 'ITA'
""").df()
model_df["is_high_frequency"] = (model_df["n_orders"] >= model_df["n_orders"].median()).astype(int)

features = pd.get_dummies(model_df.drop(columns=["customer_id"]), columns=["loyalty_tier"], drop_first=True)
X = features.drop(columns=["n_orders", "is_high_frequency"]).astype(float)
y = features["is_high_frequency"]

clf = LogisticRegression(max_iter=1000).fit(X, y)
model_df["propensity"] = clf.predict_proba(X)[:, 1]

top_37 = model_df.sort_values("propensity", ascending=False).head(37)
avg_order_value = con.execute("SELECT AVG(gross_revenue) FROM silver.fact_sales_line WHERE region_code='ITA'").fetchone()[0]
expected_conversion = 0.35
expected_revenue = len(top_37) * avg_order_value * expected_conversion
print(f"Top {len(top_37)} clients ranked by propensity -- expected incremental revenue from advisor "
      f"outreach: ${expected_revenue:,.0f} (at {expected_conversion:.0%} assumed conversion)")
top_37[["customer_id", "loyalty_tier", "is_milan_cohort", "capsule_units", "propensity"]].head(10)

Top 37 clients ranked by propensity -- expected incremental revenue from advisor outreach: $23,930 (at 35% assumed conversion)


,customer_id,loyalty_tier,is_milan_cohort,capsule_units,propensity
14544,CUST-MI-01151,Platinum,True,18.0,1.0
9446,CUST-MI-01196,Private Client,True,12.0,1.0
24101,CUST-MI-00266,Private Client,True,26.0,1.0
6271,CUST-MI-00999,Private Client,True,12.0,1.0
24915,CUST-MI-00904,Private Client,True,9.0,1.0
8300,CUST-MI-01183,Private Client,True,23.0,1.0
31417,CUST-MI-01248,Platinum,True,41.0,1.0
18648,CUST-MI-00701,Private Client,True,12.0,1.0
30443,CUST-MI-00680,Private Client,True,12.0,1.0
6377,CUST-MI-00366,Platinum,True,7.0,1.0


## Recommend

Route the ranked client list to their existing advisors for personal outreach ahead
of the Portofino capsule's wider release — this is "next-best-client," not mass
marketing: a short, high-propensity list going to the relationship each client
already has.